In [1]:
from src.db.mysql import DbConnection
from selenium.webdriver.common.by import By
from src.utils.web import get_driver
from bs4 import BeautifulSoup, PageElement

sheet_id = "1ft6DG1S-iOTI6Ps4GtuITvStyHdEZNeM2Y_752ULtjY"
conn = DbConnection()
url_id= "pgclondon2024"

In [2]:
conn.close()

In [3]:
def login(driver):
    driver.get(f'https://app.meettomatch.com/{url_id}/site/signin/')
    username = driver.find_element(by=By.NAME, value="data[User][username]")
    password = driver.find_element(by=By.ID, value="password_real")
    form = driver.find_element(by=By.NAME, value="loginForm")
    username.send_keys("pavel.dolezal@gembase.io")
    password.send_keys("d1r7p98m")
    form.submit()
    sleep(5)
    return driver


In [4]:
from time import sleep
from selenium.webdriver.remote.webelement import WebElement

from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome(ChromeDriverManager().install())

login(driver)

company_urls = []

driver.get(f"https://app.meettomatch.com/{url_id}/company/")
sleep(5)
page = 1
while True:
    sleep(5)
    print(f"Page {page}")
    company_names = driver.find_elements(by=By.CLASS_NAME, value="nameDesktop")
    for it in company_names:
        company_a: WebElement = it.find_element(by=By.TAG_NAME, value="a")
        company_url = company_a.get_attribute('href')
        if company_url is not None:
            if company_url not in company_urls:
                company_urls.append(company_url)
    next_button = driver.find_element(by=By.CLASS_NAME, value="nextPage")
    if next_button is not None:
        page += 1
        next_button.click()
    else:
        break

C:\Users\erichter\AppData\Local\Temp\ipykernel_20196\4197707800.py:7: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome(ChromeDriverManager().install())


Page 1
Page 2
Page 3
Page 4
Page 5
Page 6
Page 7
Page 8
Page 9
Page 10
Page 11
Page 12
Page 13
Page 14
Page 15
Page 16
Page 17
Page 18
Page 19
Page 20
Page 21
Page 22
Page 23
Page 24
Page 25
Page 26
Page 27
Page 28
Page 29
Page 30
Page 31
Page 32
Page 33
Page 34
Page 35
Page 36
Page 37
Page 38
Page 39
Page 40
Page 41
Page 42
Page 43
Page 44
Page 45
Page 46
Page 47
Page 48
Page 49
Page 50
Page 51
Page 52
Page 53
Page 54
Page 55


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":".nextPage"}
  (Session info: chrome=121.0.6167.185)
Stacktrace:
	GetHandleVerifier [0x00A3E123+48179]
	(No symbol) [0x009C5D01]
	(No symbol) [0x008AD72D]
	(No symbol) [0x008E572B]
	(No symbol) [0x008E586B]
	(No symbol) [0x0091C5B2]
	(No symbol) [0x00903CD4]
	(No symbol) [0x0091AA1E]
	(No symbol) [0x00903A26]
	(No symbol) [0x008DB7BC]
	(No symbol) [0x008DC62D]
	GetHandleVerifier [0x00D57C33+3299139]
	GetHandleVerifier [0x00D95BF2+3553026]
	GetHandleVerifier [0x00D90BCC+3532508]
	GetHandleVerifier [0x00AD494E+664670]
	(No symbol) [0x009D0AB4]
	(No symbol) [0x009CBF08]
	(No symbol) [0x009CC02D]
	(No symbol) [0x009BDD00]
	BaseThreadInitThunk [0x76BFFCC9+25]
	RtlGetAppContainerNamedObjectPath [0x77E77C5E+286]
	RtlGetAppContainerNamedObjectPath [0x77E77C2E+238]


In [6]:
from src import external_api

values = [['url']]
for it in company_urls:
    values.append([it])
external_api.write_sheet(sheet_id, "helsinki_urls", values)

{'spreadsheetId': '1ft6DG1S-iOTI6Ps4GtuITvStyHdEZNeM2Y_752ULtjY',
 'updatedRange': 'helsinki_urls!A1:A1352',
 'updatedRows': 1352,
 'updatedColumns': 1,
 'updatedCells': 1352}

In [8]:
company_urls = external_api.read_sheet(sheet_id, "helsinki_urls", True)
conn = DbConnection()

conn.query_safe("""DROP TABLE IF EXISTS app_temp_data.temp_nordicgames_scrapper""")

conn.query("""
CREATE TABLE app_temp_data.temp_nordicgames_scrapper (
id     int auto_increment
        primary key,
dev_id tinytext null,
url tinytext null,
content_company mediumtext null,
content_people mediumtext null )
""")

driver = webdriver.Chrome(ChromeDriverManager().install())
login(driver)
for url in company_urls:
    try:
        driver.get(url['url'])
        sleep(3)
        data_company = driver.page_source
        detail_button = driver.find_element(by=By.CLASS_NAME, value="company_attendees_people_to_meet_button")
        detail_click = detail_button.get_attribute("onclick")
        people_url = detail_click.replace("goLink('","").replace("', '_self');", "")
        driver.get(people_url)
        sleep(3)
        data_people = driver.page_source
        conn.query("""
        INSERT INTO app_temp_data.temp_nordicgames_scrapper (url, content_company, content_people) VALUES (%s, %s, %s)
        """, [url['url'], data_company, data_people])
        conn.commit()
    except Exception as err:
        conn.rollback()
driver.close()

C:\Users\erichter\AppData\Local\Temp\ipykernel_20196\2319440858.py:16: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome(ChromeDriverManager().install())


In [9]:
def parse_url(url: str):
    return url.replace("goLink('","").replace("', '_self');", "").replace("', '_blank');", "")

def parse2(content_company: str, content_people: str):
    soup = BeautifulSoup(content_company, 'html.parser')

    data = {
        'nordic_url': "",
        'company_name': "",
        'company_country': "",
        'company_employees': "",
        'company_url': "",
        'company_profile': "",
        'company_interested': "",
        'contact1_name': "",
        'contact1_url': "",
        'contact1_position': "",
        'contact1_presentation': "",
        'contact1_deal_size': "",
        'contact1_looking_for': "",
        'contact1_industry': "",
        'contact1_platform': "",
        'contact1_invest': "",
    }

    o: PageElement = soup.find(attrs={"class": "attendees_company_details_name"})
    if o is not None:
        data['company_name'] = o.text

    o = soup.find(attrs={"class": "attendees_company_spacing_company_info"})
    if o is not None:
        if " • " in o.text:
            ix = o.text.index(" • ")
            data['company_country'] = o.text[:ix]
            data['company_employees'] = o.text[ix + 3:].replace(" employees", "")
        else:
            data['company_country'] = o.text
    o = soup.find(attrs={"class": "attendees_company_info_container"})
    if o is not None:
        oc = o.find_all("div")
        for it in oc:
            span1 = it.find(attrs={"class": "attendees_contact_info_title"})
            if span1 is not None and span1.text == "Company profile:":
                span2 = it.find(attrs={"class": "people_to_meet_text"})
                data['company_profile'] = span2.text
            if span1 is not None and span1.text == "Is interested in:":
                span2 = it.find(attrs={"class": "people_to_meet_text"})
                data['company_interested'] = span2.text
    o = soup.find(attrs={"class": "company_attendees_advanced_search_buttons"})
    if o is not None:
        h = o.get("onclick")
        if h is not None:
            data['company_url'] = parse_url(h)

    o = soup.find(attrs={"class": "company_attendees_people_to_meet_button"})
    if o is not None:
        h = o.get("onclick")
        if h is not None:
            data['contact1_url'] = parse_url(h)

    soup = BeautifulSoup(content_people, 'html.parser')

    o = soup.find(attrs={"class": "attendees_company_details_name"})
    if o is not None:
        data['contact1_name'] = o.text

    o = soup.find(attrs={"class": "attendees_company_spacing_company_info"})
    if o is not None:
        o1 = o.find(attrs={"class": "people_to_meet_text"})
        if o1 is not None:
            data['contact1_position'] = o1.text

    o = soup.find_all(attrs={"class": "attendees_contact_info_title"})
    for it in o:
        if it.text == "Online presentation, profile or pitch deck":
            p = it.find_parent()
            a = p.find("a")
            if a is not None:
                h = a.get("href")
                if h is not None:
                    data['contact1_presentation'] = h
        elif it.text == "Preferred deal size":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_deal_size'] = ','.join(arr)
        elif it.text == "Looking for investment type":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_invest'] = ','.join(arr)
        elif it.text == "Industry":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_industry'] = ','.join(arr)
        elif it.text == "Platform":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_platform'] = ','.join(arr)
        elif it.text == "Looking for":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_looking_for'] = ','.join(arr)

    return data



In [10]:
from src.server.models.app_model import AppModel

rows = conn.select_all("""
SELECT n.id, n.dev_id, n.url, n.content_company, n.content_people
FROM app_temp_data.temp_nordicgames_scrapper n
""")

data_arr = []
for row in rows:
    try:
        data = parse2(row['content_company'], row['content_people'])
        for k in data:
            data[k] = data[k].removeprefix("\n").removesuffix("\n")
        data['nordic_url'] = row['url']
        data['id'] = row['id']
        data['dev_id'] = row['dev_id']
        data['dev_title'] = ""
        data['dev_url'] = ""
        data['dev_apps'] = ""
        if data['dev_id'] is not None and data['dev_id'] != 'N/A':
            row_dev = conn.select_one_or_none("""
            SELECT d.title
              FROM scraped_data.devs d
             WHERE d.dev_id = %s
            """, [data['dev_id']])
            if row_dev is not None:
                data['dev_title'] = row_dev['name']
                data['dev_url'] = row_dev['link']
                row_apps = conn.select_all("""
                SELECT a.app_id, a.title, a.installs
                  FROM scraped_data.apps a,
                       scraped_data.devs_apps da
                 WHERE da.dev_id = %s
                   AND da.app_id = a.app_id
                """, [data['dev_id']])
                arr = []
                for row_app in row_apps:
                    url = AppModel.get_app_store_url(row_app["app_id"])
                    arr.append(f"{row_app['title']} - {row_app['installs']} - {url}")
                data['dev_apps'] = '\n'.join(arr)
        data_arr.append(data)
    except Exception as err:
        print(err)

In [5]:
from src.server.models.scraper_model import ScraperModel
from src.server.models.user_quota_model import QuotaContext, QuotaPortalInternalBatchContext, QuotaPortalUserContext
from src.server.models.app_dev_searcher_model import AppStoreSearch

quota = QuotaPortalInternalBatchContext(QuotaPortalUserContext.QUOTA_GOOGLE, "nordic games scraper")
for it in data_arr:
    if it['dev_id'] is not None:
        continue
    d = AppStoreSearch.search_dev_by_name(conn, it['company_name'], quota, AppModel.STORE__GOOGLE_PLAY, 1)
    if len(d) > 0:
        print(f"Developer {it['company_name']}: {d[0]}")
        it['dev_id'] = d[0]['dev_id']
        row = conn.select_one_or_none("""
        SELECT * FROM scraped_data.devs d WHERE d.dev_id = %s
        """, [it['dev_id']])
        if row is None:
            #ScraperModel.scrap_dev(conn, it['dev_id'], AppModel.STORE__GOOGLE_PLAY)
            #row2 = conn.select_one_or_none("""
            #SELECT * FROM scraped_data.devs d WHERE d.dev_id = %s
            #""", [it['dev_id']])
            #if row2 is not None:
            conn.query("""
            UPDATE app_temp_data.temp_nordicgames_scrapper n SET n.dev_id = %s
            WHERE n.id = %s
            """, [it['dev_id'], it['id']])
        else:
            conn.query("""
            UPDATE app_temp_data.temp_nordicgames_scrapper n SET n.dev_id = %s
            WHERE n.id = %s
            """, [it['dev_id'], it['id']])
    else:
        it['dev_id'] = 'N/A'
        conn.query("""
        UPDATE app_temp_data.temp_nordicgames_scrapper n SET n.dev_id = 'N/A'
        WHERE n.id = %s
        """, [it['id']])
    conn.commit()


Developer KuuHubb Oy: {'dev_id': 'Kuu+Hubb+Oy', 'title': 'Kuu Hubb Oy', 'external': 1, 'store': 1}
Developer LALIGA: {'dev_id': 'LaLiga+Entertainment', 'title': 'LaLiga Entertainment', 'external': 1, 'store': 1}
Developer Matrioshka Games: {'dev_id': '7656930928417290829', 'title': 'Matryoshka', 'external': 1, 'store': 1}
Developer -: {'dev_id': '5813942451248631393', 'title': 'Stillfront', 'external': 0, 'store': 1}
Developer Playstudios: {'dev_id': '6266607152549785498', 'title': 'PLAYSTUDIOS INC', 'external': 0, 'store': 1}
Developer Meta: {'dev_id': 'META+PLAY', 'title': 'META PLAY', 'external': 0, 'store': 1}
Developer NetEase Games: {'dev_id': 'NetEase+Games', 'title': 'NetEase Games', 'external': 0, 'store': 1}
Developer Print Factory: {'dev_id': '5625759341548857477', 'title': 'SekGames', 'external': 1, 'store': 1}
Developer Game On Loop: {'dev_id': '5014688380385987684', 'title': 'Loop Games', 'external': 1, 'store': 1}
Developer None: {'dev_id': 'Lynkit.', 'title': 'Lynkit.',

In [11]:
values = []
if len(data_arr) > 0:
    values.append([])
    for k in data_arr[0]:
        values[0].append(k)
    for data in data_arr:
        arr = [""] * len(values[0])
        values.append(arr)
        for k in data:
            ix = values[0].index(k)
            arr[ix] = data[k]

external_api.write_sheet(sheet_id, "helsinki_data", values)

{'spreadsheetId': '1ft6DG1S-iOTI6Ps4GtuITvStyHdEZNeM2Y_752ULtjY',
 'updatedRange': 'helsinki_data!A1:U1348',
 'updatedRows': 1348,
 'updatedColumns': 21,
 'updatedCells': 26961}

In [7]:
conn.close()